# 14.12 每輪到底把什麼交回去？


訂單文字沒有變，第二輪為何能做不同的事？因為區塊讀的不只是原文字，還有上一輪留下的**隱藏狀態**：每個輸入位置的一排向量。向量是可變的工作材料，參數則在一次推論期間固定。

令 `e` 是原文查embedding表得到的向量序列，`s_r` 是第 r 輪後的狀態。最小示意寫成 `s_(r+1)=F(e,s_r;θ)`，θ 是共享區塊的參數。每輪把 e 再送入，讓更新仍可參照原問題；模型不用完全依賴上一輪把所有線索留住。輸出層 `head(s_r)` 才把狀態轉成答案分數。不同架構可以在循環前後另加不共用的區塊，或把輪數也當成輸入；本課先把共享關係看清楚。

下面是**手工狀態更新**，沒有注意力、參數學習或訓練成果。把要接近的答案向量簡化成數字4，每輪補上目前差距的一半；目的是看出輸入不變，狀態仍能改變。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
e, state = 4.0, 0.0
for round_id in range(1, 5):
    state = state + 0.5 * (e - state)
    print(round_id, state)

狀態依序是2、3、3.5、3.75。這個人寫的更新規則會靠近4，因為每次差距恰好減半。訓練出來的 Transformer 並沒有自動繼承這項性質；把「反覆處理」畫成箭頭，不能證明它也在朝正確答案收斂。

還要分清兩條軸：輸入 token 軸是「訂單、增加、取消」這些位置；loop 軸是對**同一批位置的狀態**反覆處理。跑四輪不代表新增四個輸出 token。逐 token 生成的語言模型（每次生成一個 token，接回輸入再算下一個，[1.14](https://birdhackor.github.io/tiny-perceptron-vlm/1.14.html)）仍可在產生每個 token 前做多輪；本課第19章的 Looped 解題器則讀完一整份題目後直接分類出最後答案，省去逐 token 生成的干擾，讓輪數容易量測。

原問題應否每輪再注入，也是一項可檢查的設計。若只剩已被改寫的狀態，早期丟掉的細節可能難以恢復；保留 e 讓區塊每輪都能再讀原問題，但沒有保證區塊真的使用它。真正訓練出來的模型還要用改問題、改關鍵事件的測試來確認。

練習把程式的補差距係數 `0.5` 改成 `2`。狀態會在0與8之間來回，不再接近4。相同的循環形式可以收斂，也可以振盪；下一步要看訓練目標到底監督哪一輪。
